# Module `models.py`

Ce notebook presente les structures de donnees centrales du projet CESIPATH. Ces objets sont le vocabulaire commun entre le generateur, les solveurs, la simulation dynamique, les validateurs et la visualisation.

Le point important n'est pas seulement la liste des classes : c'est la **frontiere de responsabilite**. `models.py` decrit l'etat du monde, mais ne decide pas comment le construire, le faire evoluer ou l'optimiser. Cette separation rend le projet plus facile a expliquer : quand on lit un objet, on sait quelles donnees existent ; quand on cherche une logique metier, on va dans le module specialise.

On y trouve trois familles d'objets :

- les objets d'aretes (`EdgeStatus`, `EdgeAttributes`) ;
- la configuration de generation (`GraphGenerationConfig`) ;
- les conteneurs d'etat (`GraphInstance`, `DynamicGraphSnapshot`).


## Pourquoi commencer par les modeles ?

Les modeles sont la base du contrat entre modules. Si cette couche est claire, le reste du code devient beaucoup plus simple a raisonner.

Separer les structures du comportement permet :

- aux solveurs de ne dependre que d'un `SolverInput`, construit a partir de ces modeles ;
- au generateur de fabriquer des dictionnaires d'aretes puis de les emballer dans une `GraphInstance` coherente ;
- au validateur de raisonner sur des proprietes observables (`density`, `average_degree`, connexite) sans connaitre les tirages aleatoires ;
- a la dynamique de produire des snapshots successifs sans modifier l'instance statique de depart.

La regle mentale est simple : **un modele transporte une information stable, un service la transforme**. Cela evite que des calculs caches apparaissent au mauvais endroit.


In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from cesipath.models import (
    DynamicGraphSnapshot,
    EdgeAttributes,
    EdgeStatus,
    GraphGenerationConfig,
    GraphInstance,
)

## 1. `EdgeStatus`

Enum a trois valeurs qui decrit l'etat **statique** d'une arete :

- `FREE` (`libre`) : arete utilisable, sans surcout ;
- `SURCHARGED` (`surcout`) : arete utilisable mais plus couteuse ;
- `FORBIDDEN` (`interdit`) : arete retiree du graphe residuel.

Le mot **statique** est important. Une arete `FORBIDDEN` ne fait jamais partie du reseau routier exploitable : elle est filtree avant Dijkstra. A l'inverse, une arete libre ou surchargee peut devenir temporairement indisponible en dynamique, mais ce changement est stocke dans `DynamicGraphSnapshot.edge_availability`, pas dans `EdgeStatus`.

L'enum herite de `str`. Cela rend les valeurs faciles a afficher, serialiser ou inspecter dans un notebook (`status.value` donne directement le libelle francais). On evite aussi les nombres magiques comme `0`, `1`, `2`, qui deviennent vite illisibles dans les tableaux de debug.


In [2]:
for status in EdgeStatus:
    print(status.name, "->", status.value)

FREE -> libre
SURCHARGED -> surcout
FORBIDDEN -> interdit


## 2. `EdgeAttributes`

`EdgeAttributes` decrit une vraie route physique du graphe non oriente. La dataclass est `frozen` au sens des dataclasses Python [1] : une fois creee, une arete ne change pas de cout de base ni de statut statique. Quand on veut un autre etat, on cree un nouvel objet.

Trois champs :

- `base_cost` : cout physique minimal, typiquement la distance euclidienne entre deux coordonnees ;
- `status` : valeur d'`EdgeStatus` ;
- `static_surcharge` : coefficient applique si `status=SURCHARGED`.

La propriete `static_cost` encode la logique metier :

- arete libre : `static_cost = base_cost` ;
- arete surchargee : `static_cost = base_cost * (1 + static_surcharge)` ;
- arete interdite : `static_cost = inf`.

**Pourquoi une propriete plutot qu'un champ stocke ?** Parce que `static_cost` est derive de `base_cost`, `status` et `static_surcharge`. Le recalculer a la lecture garantit qu'il reste coherent. Si on stockait les quatre valeurs, il serait possible d'avoir `status=FORBIDDEN` mais `static_cost=10`, ce qui rendrait les bugs beaucoup plus difficiles a trouver.


In [3]:
free_edge = EdgeAttributes(base_cost=10.0)
surcharged_edge = EdgeAttributes(base_cost=10.0, status=EdgeStatus.SURCHARGED, static_surcharge=0.25)
forbidden_edge = EdgeAttributes(base_cost=10.0, status=EdgeStatus.FORBIDDEN)

print("Arete libre      :", free_edge.static_cost)
print("Arete surchargee :", surcharged_edge.static_cost)
print("Arete interdite  :", forbidden_edge.static_cost)

Arete libre      : 10.0
Arete surchargee : 12.5
Arete interdite  : inf


## 3. `GraphGenerationConfig`

`GraphGenerationConfig` regroupe tous les leviers de generation. Elle contient a la fois des parametres fournis par l'utilisateur (`node_count`, `seed`, `vehicle_capacity`, probabilites dynamiques) et des seuils optionnels qui peuvent etre derives automatiquement.

Le principe : l'utilisateur donne peu, le reste est **derive** via les proprietes `resolved_*`. Exemple : si `edge_density` n'est pas fourni, la config choisit une densite cible compatible avec la taille de l'instance. Si les bornes dynamiques ne sont pas donnees, elles sont deduites des bornes residuelles.

`__post_init__` joue le role de pare-feu. Il refuse les configurations impossibles avant meme que le generateur tire une seule coordonnee : densite hors `[0, 1]`, probabilite invalide, depot hors bornes, capacite negative, intervalle min/max inverse, etc.

**Pourquoi centraliser ces validations ici ?** Parce qu'une mauvaise configuration est une erreur d'entree, pas une erreur de generation. Si on la detecte tard, on obtient des echecs ambigus du type "impossible de generer une instance" alors que le vrai probleme est simplement `min_density > max_density`.


In [4]:
config = GraphGenerationConfig(node_count=12, seed=42)
print("edge_density fournie        :", config.edge_density)
print("auto_density_profile        :", config.auto_density_profile)
print("resolved_edge_density       :", config.resolved_edge_density)
print("resolved_min_base_density   :", config.resolved_min_base_density)
print("resolved_max_base_density   :", config.resolved_max_base_density)
print("resolved_min_residual_density :", config.resolved_min_residual_density)
print("resolved_max_residual_density :", config.resolved_max_residual_density)

edge_density fournie        : None
auto_density_profile        : True
resolved_edge_density       : 0.45
resolved_min_base_density   : 0.3
resolved_max_base_density   : 0.6
resolved_min_residual_density : 0.22
resolved_max_residual_density : 0.5


## 4. Profil de densite automatique

Quand `auto_density_profile=True` et qu'aucune borne n'est specifiee, `_recommended_density_profile(node_count)` applique des fourchettes adaptees a la taille du graphe :

| `node_count` | min base | max base | min residuelle | max residuelle |
|---|---:|---:|---:|---:|
| `<= 10`  | 0.45 | 0.75 | 0.35 | 0.65 |
| `<= 25`  | 0.30 | 0.60 | 0.22 | 0.50 |
| `> 25`   | 0.18 | 0.40 | 0.12 | 0.30 |

**Pourquoi reduire la densite quand `n` grandit ?** Dans un graphe non oriente simple, le nombre maximal d'aretes vaut `n(n-1)/2` [2]. Doubler le nombre de sommets multiplie presque par quatre le nombre d'aretes possibles. Garder une densite fixe produirait tres vite des graphes trop denses, lents a afficher et moins interessants pour la fermeture metrique.

Le profil automatique garde donc un compromis : assez d'aretes pour rester connexe et robuste, mais assez peu pour que les detours, les interdictions et Dijkstra aient un vrai role.


In [5]:
for n in [6, 15, 30, 60]:
    cfg = GraphGenerationConfig(node_count=n, seed=0)
    print(
        f"n={n:>3} | base ~ [{cfg.resolved_min_base_density}, {cfg.resolved_max_base_density}]"
        f"  | res ~ [{cfg.resolved_min_residual_density}, {cfg.resolved_max_residual_density}]"
        f"  | cible = {cfg.resolved_edge_density}"
    )

n=  6 | base ~ [0.45, 0.75]  | res ~ [0.35, 0.65]  | cible = 0.6
n= 15 | base ~ [0.3, 0.6]  | res ~ [0.22, 0.5]  | cible = 0.45
n= 30 | base ~ [0.18, 0.4]  | res ~ [0.12, 0.3]  | cible = 0.29
n= 60 | base ~ [0.18, 0.4]  | res ~ [0.12, 0.3]  | cible = 0.29


## 5. Seuils derives (formules internes)

En plus des densites, la config deduit trois seuils utilises par les validateurs.

**Degre moyen residuel minimal** :

```text
resolved_min_average_residual_degree = max(2, min_residual_density * (n - 1) * 0.85)
```

On impose au moins 2 pour eviter les graphes connectes mais fragiles, comme une simple chaine. Une chaine est connexe, mais couper une arete la separe en deux morceaux ; elle n'est pas un bon support pour un reseau routier dynamique.

**Densite dynamique minimale** :

```text
resolved_dynamic_min_density = max(0.10, resolved_min_residual_density * 0.85)
```

La dynamique a le droit de retirer temporairement des aretes, mais pas de vider le reseau. Le facteur `0.85` donne une marge par rapport au statique tout en gardant une borne stricte.

**Degre moyen dynamique minimal** :

```text
resolved_dynamic_min_average_degree = max(2, resolved_min_average_residual_degree * 0.85)
```

Ce seuil evite qu'une simulation laisse quelques ponts critiques porter tout le reseau. Il complete la connexite : etre connecte ne suffit pas, il faut rester suffisamment maille.


In [6]:
cfg = GraphGenerationConfig(node_count=15, seed=0)
print("min_avg_residual_degree     :", cfg.resolved_min_average_residual_degree)
print("dynamic_min_density         :", cfg.resolved_dynamic_min_density)
print("dynamic_min_average_degree  :", cfg.resolved_dynamic_min_average_degree)

min_avg_residual_degree     : 2.62
dynamic_min_density         : 0.187
dynamic_min_average_degree  : 2.23


## 6. `GraphInstance`

`GraphInstance` est le paquet complet remis au reste du projet apres generation. Il contient :

- la configuration qui a produit l'instance ;
- les coordonnees des sommets ;
- les vraies aretes de base et residuelles ;
- les matrices `base_costs`, `residual_costs`, `completed_costs` ;
- les chemins reels `completed_paths` ;
- les demandes clients.

La distinction entre les trois matrices est essentielle :

- `base_costs` decrit le reseau physique avant contraintes ;
- `residual_costs` decrit les vraies aretes encore utilisables apres contraintes statiques ;
- `completed_costs` decrit le graphe complet vu par les solveurs, obtenu par plus courts chemins.

Les proprietes (`base_density`, `residual_density`, `residual_average_degree`, `minimum_route_count`) sont des lectures derivees. Elles evitent de recopier les memes formules dans les notebooks, les validateurs et les sorties de debug.

**Pourquoi `GraphInstance` n'est pas `frozen` ?** Les aretes elles-memes sont protegees par `EdgeAttributes`, mais l'instance reste un conteneur pratique pour les notebooks et les tests. La logique de production ne modifie pas l'instance apres generation ; la dynamique cree des `DynamicGraphSnapshot` separes.


In [7]:
from cesipath.graph_generator import GraphGenerator

instance = GraphGenerator(GraphGenerationConfig(node_count=8, seed=42)).generate()
print("Densites :", instance.base_density, instance.residual_density)
print("Degre moyen residuel :", instance.residual_average_degree)
print("Demande uniforme :", instance.uniform_demand)
print("Nombre minimal de tournees :", instance.minimum_route_count)

u, v = next(iter(instance.residual_edges))
print(f"Arete ({u}, {v}) :", instance.edge(u, v))

Densites : 0.5714285714285714 0.5
Degre moyen residuel : 3.5
Demande uniforme : 8
Nombre minimal de tournees : 2
Arete (0, 1) : EdgeAttributes(base_cost=41.48, status=<EdgeStatus.FREE: 'libre'>, static_surcharge=0.0)


## 7. `DynamicGraphSnapshot`

`DynamicGraphSnapshot` represente l'etat du reseau a un tour de simulation `step`. Il ne remplace pas `GraphInstance`; il s'ajoute par-dessus.

Attributs :

- `edge_costs` : cout courant par vraie arete dynamique. Meme une arete temporairement OFF garde son dernier cout connu ;
- `edge_availability` : dictionnaire `{(u,v): bool}`, `True` si l'arete est utilisable ce tour ;
- `residual_costs` : matrice des vraies aretes actives au tour courant ;
- `completed_costs` : fermeture metrique recalculee sur les aretes actives ;
- `completed_paths` : chemins reels correspondant a cette fermeture.

Le snapshot est volontairement complet. Un solveur dynamique peut recevoir uniquement la matrice complete via `SolverInput`, tandis qu'un module d'execution peut inspecter les chemins reels pour savoir si une route planifiee traverse une arete devenue indisponible.

**Difference cle** : `GraphInstance` dit ce qui existe dans le monde statique, `DynamicGraphSnapshot` dit ce qui est disponible maintenant.


In [8]:
from cesipath.dynamic_network import DynamicNetworkSimulator

sim = DynamicNetworkSimulator(instance, seed=42)
snap = sim.initialize_snapshot()
print("Type :", type(snap).__name__)
print("Step :", snap.step)
print("Aretes actives :", snap.active_edge_count)
print("Cout (0, 1) :", snap.edge_cost(0, 1))
print("Disponible (0, 1) ?", snap.is_available(0, 1))

Type : DynamicGraphSnapshot
Step : 0
Aretes actives : 14
Cout (0, 1) : 41.48
Disponible (0, 1) ? True


## 8. A retenir

`models.py` ne cherche pas a tout faire. Il fixe le vocabulaire : arete, configuration, instance, snapshot. Cette decomposition suit l'idee classique de separation des responsabilites entre modules [3]. Les autres modules utilisent ce vocabulaire pour produire du comportement.

Carte mentale :

- `GraphGenerationConfig` : ce qu'on demande au generateur ;
- `EdgeAttributes` : ce qu'est une route physique ;
- `GraphInstance` : le resultat statique complet ;
- `DynamicGraphSnapshot` : une photographie d'un tour dynamique.

La qualite de cette couche conditionne tout le reste. Si les modeles sont explicites, les notebooks d'algorithmes peuvent parler d'optimisation sans reexpliquer a chaque fois comment une arete est interdite, surchargee ou completee par Dijkstra.

---

## References

[1] **Python Software Foundation.** *dataclasses - Data Classes.* Python documentation. https://docs.python.org/3/library/dataclasses.html

[2] **Bondy, J. A. & Murty, U. S. R. (2008).** *Graph Theory.* Springer. https://doi.org/10.1007/978-1-84628-970-5

[3] **Parnas, D. L. (1972).** *On the criteria to be used in decomposing systems into modules.* Communications of the ACM, 15(12), 1053-1058. https://doi.org/10.1145/361598.361623
